In [313]:
import warnings
warnings.filterwarnings('ignore')

In [314]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [315]:
pd.set_option('display.max_columns', 100)

In [316]:
dem_uncont = 34
rep_uncont = 31

In [317]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,53,Democrats
1,54,Democrats
2,46,Republicans
3,54,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,52,Democrats
7997,53,Democrats
7998,49,Republicans


In [318]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,37,1,0.0125,Republicans
1,38,1,0.0125,Republicans
2,40,3,0.0375,Republicans
3,41,3,0.0375,Republicans
4,42,14,0.1750,Republicans


In [319]:
np.unique(seat_sims['seats']).shape[0]

26

In [320]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [321]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [322]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [323]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [324]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [325]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [326]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-7.940111,-14.014958,-12.202922,-12.247146,-9.927751,-13.767350,-11.795604,-5.645361,-12.721827,-17.681452,-1.465626,-13.638849,-11.473003,-11.938409,-12.508715,-9.624326,-9.929332,-12.930882,-10.336755,-7.831480,-13.223555,-9.138814,-7.316267,-9.685517,-9.374009,-14.556940,-14.136033,-8.494785,-13.368553,-13.190798,-10.677293,-12.244441,-11.898630,-8.403646,-16.426052,-15.432181,-8.247566,-11.200587,-9.081313,-12.920399,-10.913193,-9.217241,-14.328080,-10.935955,-13.169020,-13.658712,-13.271692,-15.498000,-8.018334,-11.850022,...,-11.985334,-11.278177,-10.423283,-15.476838,-7.681137,-11.801067,-13.369081,-8.243768,-12.719231,-8.569298,-16.575520,-16.610676,-10.910548,-7.610192,-17.815534,-7.475764,-16.725777,-3.447893,-12.682850,-8.945669,-11.963645,-10.323641,-4.710393,-21.909500,-3.355351,-14.755329,-12.299011,-13.627408,-8.602597,-12.683037,-11.264592,-12.957430,-11.353189,-8.362989,-17.046825,-12.868713,-11.477083,-19.091095,-10.716033,-10.241441,-11.075420,-14.648402,-10.864665,-10.770658,-12.541544,-13.606697,-15.232157,-10.329544,-13.031884,-12.554708
AK,0.545362,5.698895,-0.031117,5.612641,-0.878461,4.695952,5.118409,5.488871,-0.467014,-5.542856,10.953015,0.466183,-0.984249,6.579918,-0.736956,3.223235,5.766168,4.790477,-0.309080,2.899938,6.900804,4.377354,2.395891,5.174912,4.814332,1.556706,2.510196,3.580289,-0.848902,1.745397,3.561981,5.745706,7.097583,8.393928,2.942842,-0.063675,3.600976,0.213930,-0.923855,3.224912,-1.076149,3.673722,7.051179,4.220425,2.715065,-1.573987,1.034352,3.085768,2.268774,4.401638,...,4.090308,2.997137,10.075463,-0.352595,7.610502,2.842395,1.275870,3.649420,-1.364807,10.530714,2.401467,-0.875924,7.064844,3.335995,3.904350,1.578171,5.042506,2.105780,-0.624784,7.534534,1.649483,3.322725,8.585167,0.140128,4.157931,-2.368223,6.337320,5.055333,3.536230,1.041929,-3.155731,5.382328,-0.737715,5.978664,1.847206,3.056092,3.311431,7.331675,4.063669,4.015632,0.180251,4.334538,3.945041,0.385498,4.104849,0.827748,1.782550,0.631254,3.590664,1.735252
AR,-11.339002,-13.097131,-9.481903,-12.156472,-10.945170,-9.155249,-10.195647,-6.743752,-15.379313,-15.092753,-2.331686,-12.863450,-13.730323,-8.232696,-13.208242,-11.851801,-11.187582,-9.689420,-12.059636,-8.396213,-12.380681,-11.827952,-8.160896,-8.325504,-12.568393,-9.977107,-12.405226,-9.048103,-12.390346,-12.162799,-9.526284,-10.256768,-7.664191,-8.425670,-14.604336,-14.068461,-7.333077,-10.876479,-12.392621,-11.470172,-11.524023,-7.404704,-13.195791,-15.047115,-13.689058,-11.887465,-10.010726,-14.989395,-11.893905,-12.846411,...,-13.888398,-12.520638,-9.156749,-14.499237,-4.892839,-13.353364,-13.723098,-12.456664,-11.311029,-5.012654,-15.654916,-13.238977,-9.260070,-8.611615,-14.550347,-5.741813,-16.509871,-2.217355,-10.353246,-9.616439,-12.898466,-7.407886,-5.928742,-16.543096,-4.391174,-17.530355,-10.190402,-17.205988,-6.338765,-10.797903,-10.213491,-11.926713,-10.656415,-3.511388,-18.576536,-10.682199,-8.714510,-13.464566,-7.096279,-13.141786,-12.193866,-11.753039,-9.644358,-8.192399,-12.355306,-14.379833,-18.024032,-12.310275,-8.753806,-11.832882
CO,11.520123,12.302137,7.375648,14.226860,8.981176,12.665214,14.466744,13.358328,10.932049,7.258086,13.424921,13.912485,11.028892,16.291514,8.042464,15.773240,13.829532,11.914851,6.925709,6.193666,13.052095,18.810333,10.016180,16.551232,13.803401,11.598098,11.424382,20.174441,7.841402,13.638716,14.197893,10.579661,9.496538,10.599395,11.881880,5.927871,16.435660,11.322403,15.539842,9.325341,13.170575,12.149121,11.781341,13.718262,10.714897,11.851434,7.769387,11.233946,12.511902,9.654895,...,13.929259,8.758262,17.531493,

In [327]:
sim_corr = post_untransp.corr()

In [328]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.516827,0.740077,0.490251,0.497237,0.706608,0.531767,0.737172,0.705874,0.744570,0.738570,0.541078,0.751775,0.735743,0.523120,0.735109,0.708665,0.743835,0.702208,0.725533,0.503600,0.506878,0.506296,0.705548,0.507204,0.758500,0.507744,0.490724,0.491586,0.511962,0.557454,0.702635,0.506575,0.741235,0.739034
AK,0.516827,1.000000,0.516974,0.521435,0.525157,0.500905,0.558821,0.526969,0.505370,0.518098,0.518531,0.566392,0.524039,0.506060,0.560801,0.511560,0.499600,0.517281,0.498547,0.516073,0.561535,0.539310,0.554296,0.497969,0.551810,0.521053,0.547245,0.549559,0.538382,0.541739,0.597678,0.508124,0.544347,0.520646,0.519500
AR,0.740077,0.516974,1.000000,0.487471,0.499996,0.703901,0.527682,0.736311,0.698991,0.741642,0.738048,0.537178,0.742278,0.729475,0.523756,0.729408,0.700617,0.725888,0.694176,0.721901,0.509390,0.500481,0.501561,0.705674,0.504684,0.744311,0.501842,0.497013,0.495430,0.515388,0.557064,0.702387,0.498878,0.735949,0.735080
CO,0.490251,0.521435,0.487471,1.000000,0.708952,0.654635,0.550455,0.497582,0.652944,0.503369,0.488006,0.549407,0.503939,0.483418,0.538078,0.487451,0.642970,0.493861,0.653719,0.477955,0.710876,0.715793,0.719364,0.647703,0.706154,0.499915,0.710488,0.702430,0.699020,0.530764,0.571425,0.656250,0.703960,0.496860,0.494059
DE,0.497237,0.525157,0.499996,0.708952,1.000000,0.667778,0.542927,0.504032,0.663931,0.508841,0.499961,0.547754,0.503502,0.500091,0.549078,0.501816,0.663438,0.499234,0.664746,0.489481,0.719132,0.725936,0.728298,0.661533,0.715898,0.515668,0.710036,0.712796,0.709214,0.521138,0.577942,0.661885,0.719020,0.499426,0.509313


In [329]:
post.shape

(35, 8000)

In [330]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [331]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:06<00:00, 1256.97it/s]


array(['AK', 'MI', 'TX', ..., 'IA', 'IA', 'AK'],
      shape=(8000,), dtype='<U32')

In [332]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.0500,1,31.823664,45.706797
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.8000,2,46.562280,60.032036
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.1250,3,32.005690,46.006760
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.986286,47.730058,1,95.460117,61.874444,3.504022,99.9625,4,55.086058,68.778487
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.079539,48.524126,1,97.048252,63.417671,3.504467,99.9875,5,56.579388,70.374188


In [333]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.0500,1,31.823664,45.706797,0.000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.8000,2,46.562280,60.032036,5.075
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.1250,3,32.005690,46.006760,0.000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.986286,47.730058,1,95.460117,61.874444,3.504022,99.9625,4,55.086058,68.778487,0.175
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.079539,48.524126,1,97.048252,63.417671,3.504467,99.9875,5,56.579388,70.374188,0.750


In [334]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",14408058.06,6831818.66,21239876.72,67.834942,32.165058,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-9.05715,0,0,47.714292,8.029924,44.727999,8.029924,East North Central,2026,0.0,0.00000,0.00000,2,Senate,4601.579360,8.029924,2.833712,-2.986293,8.672238,17.834942,0,35.669884,50.919210,3.537732,60.1125,16,44.146135,57.914765,11.4250
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",65585741.88,9020177.98,74605919.86,87.909568,12.090432,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-9.05715,0,0,48.330668,6.556056,45.576386,6.556056,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7728.092153,6.556056,2.560480,-2.754282,-2.776598,37.909568,0,75.819136,51.122177,3.627371,62.1375,32,44.024749,58.325635,11.3125
9,9,Iowa,Josh Turek,Ashley Hinson,False,False,R,IA,"TUREK, JOSHUA","ARENHOLZ, ASHLEY HINSON",5835816.67,5267628.55,11103445.22,52.558612,47.441388,-6.092833,43.277201,88.847514,1.696738,4.401391,3.028533,0.168574,31.423453,2014831,1175538,3190369,63.153541,36.846459,0.987401,-9.05715,0,0,46.647197,5.240247,46.135733,5.240247,West North Central,2026,0.0,0.00000,0.00000,3,Senate,2762.407675,5.240247,2.289159,-0.511465,-3.128517,2.558612,0,5.117224,49.679267,3.515302,45.5375,10,42.688491,56.559194,9.6750
24,24,Ohio,Sherrod Brown,Jon Husted,False,True,R,OH,"BROWN, SHERROD","HUSTED, JON",31146133.40,6280012.05,37426145.45,83.220254,16.779746,-5.266554,44.343385,80.925356,1.708882,3.291326,11.486903,0.076606,31.487184,9001099,2798349,11799448,76.284069,23.715931,0.987401,-9.05715,0,1,48.768743,2.423055,44.102599,2.423055,East North Central,2026,0.0,0.00000,0.00000,2,Senate,6925.610718,2.423055,1.556616,-4.666143,-1.475958,33.220254,-1,66.440509,51.054280,3.466222,62.1875,25,44.140287,57.909156,9.3250
28,28,South Carolina,Annie Andrews,Darline Graham,False,False,R,SC,"ANDREWS, ANNIE","GRAHAM, DARLINE",10677209.93,299782.64,10976992.57,97.268991,2.731009,-8.282832,40.938317,67.144996,1.378347,4.124393,24.827438,0.196629,32.129153,3477869,1640556,5118425,67.948031,32.051969,0.987401,-9.05715,0,0,44.596019,1.458879,45.788282,1.458879,South Atlantic,2026,0.0,0.00000,0.00000,5,Senate,9461.256637,1.458879,1.207841,1.192263,-7.508515,47.268991,0,94.537982,49.502699,3.547292,44.2375,29,42.505162,56.446262,8.5375


In [335]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [336]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.000000,0.0,0.000000,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.0500,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.000000,0.0,0.000000,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.8000,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.000000,0.0,0.000000,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.1250,3,32.005690,46.006760,0.000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",5806791.50,134872.30,5941663.80,97.730058,2.269942,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9551.164309,0.000000,0.000000,0.000000,20.986286,47.730058,1,95.460117,61.874444,3.504022,99.9625,4,55.086058,68.778487,0.175,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",1989694.63,29805.27,2019499.90,98.524126,1.475874,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-9.05715,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9707.003443,0.000000,0.000000,0.000000,25.079539,48.524126,1,97.048252,63.417671,3.504467,99.9875,5,56.579388,70.374188,0.750,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [337]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.050,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.800,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.125,3,32.005690,46.006760,0.000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [338]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.05,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.80,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [339]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.050,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.800,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.125,3,32.005690,46.006760,0.000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.8


In [340]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.050,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.950,0.0,100.0,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.800,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,17.200,82.8,17.2,82.8%,17.2%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.125,3,32.005690,46.006760,0.000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.8,99.875,0.1,99.9,<1%,>99%


In [341]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.050,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.950,0.0,100.0,<1%,>99%,8.155987
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.800,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,17.200,82.8,17.2,82.8%,17.2%,20.144535
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",2247891.01,3174164.47,5422055.48,41.458281,58.541719,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-9.05715,0,1,38.189521,0.354297,49.190029,0.354297,West South Central,2026,0.0,0.0,0.0,5,Senate,1718.789084,0.354297,0.595229,11.000508,-21.563893,-8.541719,-1,-17.083438,39.106754,3.567951,0.125,3,32.005690,46.006760,0.000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+21.8,99.875,0.1,99.9,<1%,>99%,9.553494


In [342]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",9911.40,2615610.48,2625521.88,0.377502,99.622498,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-9.05715,0,0,32.999181,0.560745,53.994266,0.560745,East South Central,2026,0.0,0.0,0.0,5,Senate,0.142508,0.560745,0.748829,20.995085,-20.569292,-49.622498,0,-99.244996,38.637381,3.540086,0.05,1,31.823664,45.706797,0.000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+22.7,99.95,0.0,100.0,<1%,>99%,8.155987,D+8.2
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DANIEL J",16257753.91,8435.00,16266188.91,99.948144,0.051856,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-9.05715,0,1,48.974227,2.806404,47.234091,2.806404,Pacific,2026,0.0,0.0,0.0,5,Senate,9989.631483,2.806404,1.675232,-1.740137,-3.853455,49.948144,-1,99.896288,53.225705,3.403738,82.80,2,46.562280,60.032036,5.075,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.5,17.20,82.8,17.2,82.8%,17.2%,20.144535,D+20.1


In [343]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+22.7,D+8.2,0.000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">82.8%</p>","<p style=""color:red;"">17.2%</p>",D+6.5,D+20.1,5.075
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+21.8,D+9.6,0.000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+23.7,D+12.5,0.175
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.8,D+11.9,0.750


In [344]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

7.218729973309594

In [345]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,51.46650
1,chamber_win_chance,59.27500
0,sv_bias,7.21873


In [346]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [347]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,59.275,51.4665
1,Republicans,40.725,48.5335


In [348]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [349]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [350]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-10-06,59.2750,US Senate,chance,Tossup,2026-10-05,59.3625,chance,Tossup
1,2026-10-06,0.0500,AL,chance,Safe R,2026-10-05,0.0500,chance,Safe R
2,2026-10-06,82.8000,AK,chance,Likely D,2026-10-05,82.9625,chance,Likely D
3,2026-10-06,0.1250,AR,chance,Safe R,2026-10-05,0.1250,chance,Safe R
4,2026-10-06,99.9625,CO,chance,Safe D,2026-10-05,99.9625,chance,Safe D


In [351]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

   geo       rating_y rating_t
16  MI         Tossup   Tilt D
21  NH  Very Likely D   Safe D


In [352]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

(2, 3)


In [353]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')